In [1]:
from fasta import FASTA
from src.util import load_region_modifications
from src.files.files import get_files
import numpy as np
from collections import defaultdict

window_sizes = [100, 150, 200, 250, 300]
window_multipliers = [1, 2]

def get_start(modification, window_size, window_multiplier):
    return modification - int(window_size * window_multiplier)

def get_end(modification, window_size, window_multiplier):
    return modification + int(window_size * window_multiplier)

savings = defaultdict(dict)

for key, file in get_files().get_assembled_region_fasta_files().get_files_dict().items():
    savings[key] = defaultdict(dict)
    
    regions = load_region_modifications(get_files().get_assembled_region_local_intersects_assembled_files().get_files_dict()[key])
    
    file.open_or_recompute()
    fasta_file = FASTA(file.get_possibly_gzip_path())

    for window_size in window_sizes:
        savings[key][window_size] = defaultdict(dict)
        
        for window_multiplier in window_multipliers:
            savings[key][window_size][window_multiplier] = defaultdict(list)
    
    for entry in fasta_file:
        region_name = entry.name[:entry.name.find(":")]

        if not region_name in regions or len(regions[region_name].modifications) == 0:
            continue
        
        sequence = str(entry.seq)

        sequence_length = len(sequence)

        modifications = regions[region_name].modifications
        modifications.sort()

        for window_size in window_sizes:
            for window_multiplier in window_multipliers:
                total_saving = 0
                savings[key][window_size][window_multiplier]["length"].append(sequence_length)

                first_start = max(0, get_start(modifications[0], window_size, window_multiplier))
                last_end = min(len(sequence), get_end(modifications[-1], window_size, window_multiplier))

                start_saving = first_start
                savings[key][window_size][window_multiplier]["start"].append(start_saving)
                total_saving += start_saving

                end_saving = len(sequence) - last_end
                savings[key][window_size][window_multiplier]["end"].append(end_saving)
                total_saving += end_saving

                inner_saving = 0
                for i in range(len(modifications) - 1):
                    current_end = get_end(modifications[i], window_size, window_multiplier)
                    next_start = get_start(modifications[i + 1], window_size, window_multiplier)

                    if next_start <= current_end:
                        continue

                    saving = next_start - current_end

                    inner_saving += saving
                
                savings[key][window_size][window_multiplier]["inner"].append(inner_saving)

                total_saving += inner_saving
                
                savings[key][window_size][window_multiplier]["total"].append(total_saving)
                
print("done")

done


In [ ]:
import matplotlib.pyplot as plt

for key, key_savings in savings.items():
    for window_size, kw_savings in key_savings.items():
        fig, axs = plt.subplots(len(window_multipliers), 4, figsize=(15, 3))

        for i in range(len(window_multipliers)):
            window_multiplier = window_multipliers[i]

            kwm_savings = kw_savings[window_multiplier]

            axs[i][0].scatter(kwm_savings["length"], kwm_savings["start"])
            axs[i][0].set_title("Saving " + key + " " + str(window_size) + " " + str(window_multiplier) + " start")
            axs[i][1].scatter(kwm_savings["length"], kwm_savings["end"])
            axs[i][1].set_title("Saving " + key + " " + str(window_size) + " " + str(window_multiplier) + " end")
            axs[i][2].scatter(kwm_savings["length"], kwm_savings["inner"])
            axs[i][2].set_title("Saving " + key + " " + str(window_size) + " " + str(window_multiplier) + " inner")
            axs[i][3].scatter(kwm_savings["length"], kwm_savings["total"])
            axs[i][3].set_title("Saving " + key + " " + str(window_size) + " " + str(window_multiplier) + " total")
        
        fig.show()

In [4]:
from fasta import FASTA
from src.util import load_region_modifications
from src.files.files import get_files
from mypython.algorithms import accessibility, looptypes
import numpy as np
from collections import defaultdict
from pathlib import Path
import gzip
import csv

window_sizes = [150, 200, 350, 300]
footprints = [5, 10, 15, 20, 25, 30]
L = 100

def get_naive_start(modification, window_size):
    return modification - window_size

def get_naive_end(modification, window_size):
    return modification + window_size

def get_interval_from_modification(modification, window_size, sequence_length):
    interval_start = max(0, get_naive_start(modification, window_size))
    interval_end = min(sequence_length, get_naive_end(modification, window_size))

    return (interval_start, interval_end, [modification])

def are_intervals_overlapping(a, b):
    return a[1] >= b[0]

def combine_intervals(a, b):
    return (a[0], b[1], a[2] + b[2])

done_keys = []


for key, file in get_files().get_assembled_region_fasta_files().get_files_dict().items():
    key_regions = 0
    key_intervals = 0
    key_work = 0
    
    if key in done_keys:
        continue
        
    print(key)
    regions = load_region_modifications(get_files().get_assembled_region_local_intersects_assembled_files().get_files_dict()[key])
    
    file.open_or_recompute()
    fasta_file = FASTA(file.get_possibly_gzip_path())
    
    for i, entry in enumerate(fasta_file):        
        region_name = entry.name[:entry.name.find(":")]

        if not region_name in regions or len(regions[region_name].modifications) == 0:
            continue
        
        sequence = str(entry.seq).upper().replace("T", "U")

        sequence_length = len(sequence)

        modifications = regions[region_name].modifications
        modifications.sort()

        for window_size in window_sizes:
            intervals = []

            for modification in modifications:
                intervals.append(get_interval_from_modification(modification, window_size, sequence_length))

            i = 0

            while i < len(intervals) - 1:
                interval_a = intervals[i]
                interval_b = intervals[i + 1]
                
                if are_intervals_overlapping(interval_a, interval_b):
                    combined_interval = combine_intervals(interval_a, interval_b)
                    intervals[i] = combined_interval
                    intervals.pop(i + 1)
                    continue
                
                i += 1

            for start, end, interval_modifications in intervals:
                key_intervals += 1
                interval_sequence = sequence[start:end]

                interval_modifications_str = [str(x) for x in interval_modifications]

                adjusted_interval_modifications = [x - start for x in interval_modifications]

                for footprint in footprints:
                    for feature in looptypes.keys():
                        key_work += 1
    print(key, "done", key_work)
print("done")

3utr
3utr done 531120
5utr
5utr done 253008
5utr_start
5utr_start done 272040
cds
cds done 523128
coding_exons
coding_exons done 759024
coding_introns
coding_introns done 1241736
exons
exons done 870960
introns
introns done 1422528
non_coding_exons
non_coding_exons done 111936
non_coding_introns
non_coding_introns done 180792
done
